In [1]:
# package import
import chipwhisperer as cw
from cw_plugins.scopes import *
from cw_plugins.targets import *
import gui_helper as gui


## Scope Setup
### Select a VISA Oscilloscope
This notebook uses an external VISA-compatible oscilloscope supported by `cw_plugins.scopes`. Run the next cell and select the instrument's VISA resource address. For a ChipWhisperer Lite or Husky capture board, use [acquire_traces_cw_scope.ipynb](acquire_traces_cw_scope.ipynb).


In [ ]:
gui.showScopeSelector()

### Connect to the Selected Oscilloscope
After selecting an instrument, run the next cell to create `scope`. Select an actual instrument resource rather than `None`.


In [ ]:
# for VISA oscilloscope
scope = Oscilloscope(gui.get_visa_address())

### Sampling Rate
Run the next cell to display the sampling-rate controls. Enter a value, select `GSa/s`, `MSa/s`, or `kSa/s`, and click **apply**. Available rates depend on the connected oscilloscope; refer to its manual for supported values.


In [ ]:
gui.showSamplingRateConfig(scope)

### Trigger Configuration
Run the next cell to display the trigger controls. Select the edge mode (`rising`, `falling`, or `either`) and the channel carrying the trigger signal, then enter its vertical **Scale** and **Offset** in the selected voltage units. Scale is volts per division; Offset is a voltage.

Use different channels for the trigger signal and power trace. Click **apply** to send the settings to the oscilloscope; editing a field requires applying again.


In [ ]:
gui.showTriggerConfig(scope)

### Trace Configuration
Run the next cell to display the trace controls. Select the channel carrying the power trace, then set:

- **Scale**: vertical scale per division, in `V` or `mV`.
- **Offset**: vertical offset, in `V` or `mV`.
- **Period**: waveform duration, with `s`, `ms`, `us`, or `ns` units.
- **Delay**: capture delay relative to the trigger, in the selected time units. A negative delay requests pretrigger acquisition where supported by the oscilloscope.

Choose a trace channel different from the trigger channel and a positive Period. Click **apply** to send the settings to the oscilloscope. Channel numbers, available durations, and delays depend on the instrument.


In [ ]:
gui.showTraceConfig(scope)

## Target Hardware Setup
Run the next cell to display the target settings panel. Select `SAKURA-X` or `CW305`, then choose the implementation that matches the FPGA design:

- `AES RTL`: AES-128 RTL core; select `aist`, `google`, or `rsm` under **Select RTL Core**.
- `AES HLS`: AES-128 implementation generated by high-level synthesis.
- `AES VexRiscV`: AES-128 software running on VexRiscV; **Enable Masking** selects the masked version.

### SAKURA-X
Program the board with the matching shell and target design before connecting. Select both **Data Port** (FT2232H Channel A) and **Reset Port** (Channel B). These must be different ports belonging to the same board.

Alternatively, enable **Auto detect** on Linux with the repository's udev rules installed and one SAKURA-X board connected. On other platforms, select both ports explicitly. The GUI does not program the SAKURA-X FPGA.

### CW305
If you do not want to use the preinstalled bitstream, use **Bitfile** to select a `.bit` file and **HWH file** to select the corresponding `.hwh` memory-map file. For a custom design, provide matching files and select the matching implementation. If files are omitted, the target driver uses its default file selection; the bundled examples include prebuilt bitstreams and HWH files.

See the [hardware documentation](../docs/hardware.md) for board preparation and target-specific details.


In [ ]:
gui.showBoardSettingsPanel()

### Connect to the Target
After completing the settings above, run the next cell to connect the selected target and assign it to `target`. Finish scope configuration before starting capture.

For SAKURA-X, `gui.connectBoard(scope)` does not perform a hardware reset. If the target is stuck, call `target.hard_reset()` after connecting and before capture to reset the shell controller and target hardware.


In [ ]:
target = gui.connectBoard(scope)

## Trace Capture and Save
Run the next cell to display the capture panel.

- **Number of Traces**: number of successful traces to collect (default: 100).
- **Draw Interval**: redraw the waveform after this many successful traces (default: 5).
- **Change Key**: generate a new key before capture. The displayed key is fixed throughout a capture; plaintexts are generated for each trace.

Click **Start Capture** to acquire traces. Failed captures are retried until the requested number of successful traces is reached. Each plotted waveform is the most recently collected trace.

When capture finishes, select a save directory, enter **Project Name**, and click **Save Project**. Enable **Overwrite if exists** to replace an existing project. Saving creates a ChipWhisperer `.cwp` project and its associated data directory. Save the current capture before starting another capture, which creates a new in-memory project.


In [ ]:
gui.showCapturePanel(scope, target)